# 00 · Setup y visión general de los datos

**Proyecto:** Mercado Automotriz en La Habana (ICD)
**Fase:** 4.1 — Verificación de datasets y preparación del EDA

Este notebook solo **carga y describe** los tres conjuntos de datos del proyecto.
El análisis exploratorio completo se desarrollará en los notebooks siguientes, siguiendo `docs/eda_plan.md`.

| Dataset | Archivo | Descripción |
|---|---|---|
| Mercado informal | `data/processed/anuncios_revolico_final.json` | Anuncios de venta de Revolico |
| Mercado formal | `data/processed/mercado_formal.json` | Concesionarios e importadoras |
| Entrevistas | `data/raw/entrevistas.json` | Declaraciones de compradores y vendedores |

## 1. Librerías y configuración

In [1]:
import json
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 140)

print("pandas", pd.__version__)
print("seaborn", sns.__version__)
print("matplotlib", plt.matplotlib.__version__)

pandas 3.1.1
seaborn 0.13.2
matplotlib 3.11.2


## 2. Carga de los tres datasets

Se leen los JSON en UTF-8 y se comprueba el número de registros de cada archivo.

In [2]:
BASE = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = BASE / "data" / "raw"
DATA_PROCESSED = BASE / "data" / "processed"


def cargar_json(ruta):
    """Carga un archivo JSON en UTF-8."""
    with open(ruta, encoding="utf-8") as archivo:
        return json.load(archivo)


revolico = cargar_json(DATA_PROCESSED / "anuncios_revolico_final.json")
formal = cargar_json(DATA_PROCESSED / "mercado_formal.json")
entrevistas = cargar_json(DATA_RAW / "entrevistas.json")

print(f"Mercado informal (Revolico): {len(revolico)} anuncios")
print(f"Mercado formal: {len(formal)} entidades")
print(f"Entrevistas: {len(entrevistas)} registros")

Mercado informal (Revolico): 30 anuncios
Mercado formal: 10 entidades
Entrevistas: 30 registros


## 3. Mercado informal — anuncios de Revolico

Cada registro describe un anuncio: precio, moneda, ubicación, marca, modelo, año, tipo de vehículo y motorización.

In [3]:
df_revolico = pd.DataFrame(revolico)
print("Dimensiones:", df_revolico.shape)
print("\nColumnas y tipos de dato:")
print(df_revolico.dtypes.to_string())

df_revolico[[
    "id_anuncio", "marca", "modelo", "ano", "tipo_vehiculo",
    "tipo_motorizacion", "precio_anuncio", "moneda_anuncio", "precio_MN", "ubicacion",
]].head()

Dimensiones: (30, 12)

Columnas y tipos de dato:
id_anuncio             str
url                    str
titulo_original        str
precio_anuncio       int64
moneda_anuncio         str
precio_MN            int64
ubicacion              str
marca                  str
modelo                 str
tipo_vehiculo          str
ano                  int64
tipo_motorizacion      str


,id_anuncio,marca,modelo,ano,tipo_vehiculo,tipo_motorizacion,precio_anuncio,moneda_anuncio,precio_MN,ubicacion
0,49986625,Hyundai,I10,2010,Hatchback,combustible,12000,USD,11040000,"La Habana, La Lisa"
1,54821295,Hyundai,Palisade,2021,SUV,combustible,43500,USD,40020000,"La Habana, Arroyo Naranjo"
2,54821368,Hyundai,Elantra,2021,Sedán,combustible,29500,USD,27140000,"La Habana, Arroyo Naranjo"
3,56066787,Hyundai,Kona,2022,SUV,combustible,34000,USD,31280000,"La Habana, Arroyo Naranjo"
4,56069281,Hyundai,Auto Carro Moderno,2021,SUV,combustible,45000,USD,41400000,"La Habana, Arroyo Naranjo"


## 4. Mercado formal — concesionarios e importadoras

El archivo contiene entidades con campos anidados (`precios`, `fuentes`). `json_normalize` aplana el primer nivel y conserva las listas como objetos.

In [4]:
df_formal = pd.json_normalize(formal)
print("Dimensiones:", df_formal.shape)
print("\nColumnas:")
for columna in df_formal.columns:
    print(" -", columna)

df_formal[["id_entidad", "nombre", "tipo_entidad", "marcas_comercializadas"]].head(10)

Dimensiones: (10, 23)

Columnas:
 - id_entidad
 - nombre
 - nombre_normalizado
 - tipo_entidad
 - marcas_comercializadas
 - modelos_disponibles
 - tipos_vehiculos
 - vehiculos_electricos
 - informacion_importacion
 - precios
 - ubicacion
 - sitio_web
 - contacto_publico
 - descripcion
 - fuentes
 - fecha_ultima_verificacion
 - nivel_confidencia
 - observaciones
 - enriquecimiento_catalogos.fecha
 - enriquecimiento_catalogos.fuente
 - enriquecimiento_catalogos.campos_actualizados
 - enriquecimiento_catalogos.tipos_vehiculos_previos
 - enriquecimiento_catalogos.modelos_previos


,id_entidad,nombre,tipo_entidad,marcas_comercializadas
0,BDC_ONE,BDC ONE,concesionario,"[MG, Maxus, Mitsubishi]"
1,DONGFENG_CUBA,Dongfeng Cuba,otro,[Dongfeng]
2,MCV_COMERCIAL,"MCV Comercial, S.A.",comercializadora,[]
3,SASA,"Servicios Automotores, S.A. (SASA)",comercializadora,[]
4,DIVEP,Empresa Comercializadora DIVEP,comercializadora,[]
5,IMPEXPORT,"IMPEXPORT, S.U.R.L.",importadora,[]
6,CIMEX_DTA,"Corporación CIMEX, S.A. División de Transporte...",comercializadora,[]
7,PALCO,Empresa Importadora PALCO,importadora,[]
8,TRANSIMPORT,Empresa Central de Abastecimiento y Venta de E...,importadora,[]
9,TRADEX,Empresa Proveedora General del Transporte (TRA...,importadora,[]


In [5]:
resumen_formal = pd.DataFrame({
    "entidad": df_formal["nombre"],
    "tipo": df_formal["tipo_entidad"],
    "marcas": df_formal["marcas_comercializadas"].apply(len),
    "modelos": df_formal["modelos_disponibles"].apply(len),
    "electricos": df_formal["vehiculos_electricos"].apply(len),
})
resumen_formal

,entidad,tipo,marcas,modelos,electricos
0,BDC ONE,concesionario,3,16,5
1,Dongfeng Cuba,otro,1,49,16
2,"MCV Comercial, S.A.",comercializadora,0,0,0
3,"Servicios Automotores, S.A. (SASA)",comercializadora,0,0,0
4,Empresa Comercializadora DIVEP,comercializadora,0,0,0
5,"IMPEXPORT, S.U.R.L.",importadora,0,0,0
6,"Corporación CIMEX, S.A. División de Transporte...",comercializadora,0,0,0
7,Empresa Importadora PALCO,importadora,0,0,0
8,Empresa Central de Abastecimiento y Venta de E...,importadora,0,0,0
9,Empresa Proveedora General del Transporte (TRA...,importadora,0,0,0


## 5. Entrevistas

Declaraciones estructuradas de personas sobre compra y venta de autos en La Habana.

In [6]:
df_entrevistas = pd.DataFrame(entrevistas)
print("Dimensiones:", df_entrevistas.shape)
print("\nColumnas y tipos de dato:")
print(df_entrevistas.dtypes.to_string())

df_entrevistas.head()

Dimensiones: (30, 7)

Columnas y tipos de dato:
id_entrevista                          str
zona                                   str
ocupacion                              str
relacion_con_el_mercado                str
propietario_vehiculo                   str
experiencia_vehiculos_electricos       str
respuestas                          object


,id_entrevista,zona,ocupacion,relacion_con_el_mercado,propietario_vehiculo,experiencia_vehiculos_electricos,respuestas
0,E01,Cerro,Mecánico independiente,Repara autos en su casa-taller,Sí (Lada 2107),No,"{'tipo_vehiculo_observado': 'Clásicos', 'marca..."
1,E02,Alamar,Cuentapropia,Un familiar posee auto,No,No,"{'tipo_vehiculo_observado': 'Clásicos', 'marca..."
2,E03,Alamar,Taxista particular,Taxista,Sí (Chevrolet 1955(almendrón)),No,"{'tipo_vehiculo_observado': 'Clásicos', 'marca..."
3,E04,Alamar,Joven profesional (informático),Considera comprar su primer auto,No,Sí (ha visto uno de cerca),"{'tipo_vehiculo_observado': 'Clásicos', 'marca..."
4,E05,Alamar,Mipyme (mensajería),Ha importado vehículos para su negocio,No (camioneta asignada),No,"{'tipo_vehiculo_observado': 'Modernos', 'marca..."


## 6. Revisión rápida de valores nulos

Antes del EDA conviene saber qué campos tienen información incompleta. Los nulos no se corrigen aquí: se documentan.

In [7]:
def resumen_nulos(df, nombre):
    """Imprime los campos con valores nulos de un DataFrame."""
    nulos = df.isna().sum()
    nulos = nulos[nulos > 0].sort_values(ascending=False)
    print(f"--- {nombre} ({len(df)} filas) ---")
    if nulos.empty:
        print("Sin valores nulos.")
    else:
        print(nulos.to_string())
    print()


resumen_nulos(df_revolico, "Anuncios Revolico")
resumen_nulos(df_formal, "Mercado formal")
resumen_nulos(df_entrevistas, "Entrevistas")

--- Anuncios Revolico (30 filas) ---
Sin valores nulos.

--- Mercado formal (10 filas) ---
enriquecimiento_catalogos.fecha                      8
enriquecimiento_catalogos.fuente                     8
enriquecimiento_catalogos.tipos_vehiculos_previos    8
enriquecimiento_catalogos.campos_actualizados        8
enriquecimiento_catalogos.modelos_previos            8
contacto_publico                                     2

--- Entrevistas (30 filas) ---
Sin valores nulos.



## 7. Comprobaciones de integridad

Verificación mínima antes de analizar: identificadores únicos, rangos de precios y presencia de vehículos eléctricos.

In [8]:
print("IDs únicos en Revolico:", df_revolico["id_anuncio"].nunique(), "de", len(df_revolico))
print("Precios USD:", df_revolico["precio_anuncio"].min(), "-", df_revolico["precio_anuncio"].max())
print("Anuncios eléctricos:", int((df_revolico["tipo_motorizacion"] == "electrico").sum()))
print("Entidades con catálogo cargado:", int((df_formal["modelos_disponibles"].apply(len) > 0).sum()))
print("Modelos formales registrados:", int(df_formal["modelos_disponibles"].apply(len).sum()))
print("Entrevistas con ID único:", df_entrevistas["id_entrevista"].nunique() == len(df_entrevistas))

IDs únicos en Revolico: 30 de 30
Precios USD: 10000 - 100000
Anuncios eléctricos: 3
Entidades con catálogo cargado: 2
Modelos formales registrados: 65
Entrevistas con ID único: True


## 8. Siguientes pasos

Este notebook **no** realiza análisis profundo. El plan completo está en `docs/eda_plan.md` e incluye:

1. Distribución de precios del mercado informal (USD y MN).
2. Marcas y tipos de vehículo más frecuentes.
3. Eléctricos vs. combustión: oferta y precios.
4. Comparación formal–informal (modelos, motorización, precios).
5. Cruce con las entrevistas (motivaciones, barreras y expectativas).

Los hallazgos de los catálogos formales están documentados en `reports/catalogos_analisis_extra.md`.